In [ ]:
import random
import uuid
import string
import json
import time
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import parse_json, col, lit
from snowflake.snowpark.types import StructType, StructField, StringType

session = get_active_session()

In [ ]:
%%sql -r setup_db
CREATE OR REPLACE DATABASE STREAM;
CREATE OR REPLACE  SCHEMA STREAM.PUBLIC;

In [ ]:
%%sql -r create_table_result
CREATE OR REPLACE ICEBERG TABLE STREAM.PUBLIC.STREAM1 (
    DATA VARIANT
)
    CATALOG = 'SNOWFLAKE'
    EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
    ICEBERG_VERSION = 3;

The notebook generates synthetic web analytics / clickstream event data and inserts it into a Snowflake Iceberg table (STREAM.PUBLIC.STREAM1). 
<br>Here's a breakdown:

Data Structure
Each record is a single VARIANT column (DATA) containing a JSON object with these fields:

custom_properties
Marketing and experimentation metadata:

country — one of 10 countries (US, FR, DE, GB, JP, BR, IN, CA, AU, MX)
experiment_id — random A/B test identifier (e.g. exp_a3f1b2)
utm_campaign — marketing campaign type (retargeting, brand_awareness, conversion, loyalty, seasonal)
utm_medium — traffic channel (social, email, cpc, organic, referral)
utm_source — traffic source (email, facebook, google, twitter, linkedin, instagram)
variant — A/B test variant (control, variant_a, variant_b, variant_c)
data
A random hex string (128 chars) — acts as a payload/placeholder.

device_context
Client device information:

browser — Chrome, Firefox, Safari, Edge, Opera
browser_version — version 100–125
device_type — desktop, mobile, tablet
os — various Windows, macOS, Linux, iOS, Android, ChromeOS versions
screen_resolution — common screen sizes
user_agent — static placeholder string
event_context
The behavioral event itself:

event_type — weighted distribution: click (70%), add_to_cart (25%), purchase (5%), plus page_view, form_submit, scroll, sign_up
tags — random subset from: trending, recommended, premium, new, featured, popular, sale, limited
timestamp — randomized time within the current day (ISO 8601 UTC)
identifiers
event_id — unique UUID per event
session_id — unique UUID per session
Scale & Storage
10 million rows per run (configurable via row_count)
Stored as an Iceberg v3 table with Snowflake-managed catalog and external volume
Warehouse is scaled up to LARGE for the insert, then back to MEDIUM
Purpose
This simulates a realistic clickstream event stream — useful for testing streaming ingestion pipelines, analytics dashboards, or A/B testing frameworks at scale.

In [ ]:
COUNTRIES = ["US", "FR", "DE", "GB", "JP", "BR", "IN", "CA", "AU", "MX"]
UTM_CAMPAIGNS = ["retargeting", "brand_awareness", "conversion", "loyalty", "seasonal"]
UTM_MEDIUMS = ["social", "email", "cpc", "organic", "referral"]
UTM_SOURCES = ["email", "facebook", "google", "twitter", "linkedin", "instagram"]
VARIANTS = ["control", "variant_a", "variant_b", "variant_c"]
BROWSERS = ["Chrome", "Firefox", "Safari", "Edge", "Opera"]
DEVICE_TYPES = ["desktop", "mobile", "tablet"]
OS_LIST = [
    "Windows 10", "Windows 11", "Windows Server 2022",
    "macOS 12", "macOS 13", "macOS 14", "macOS 15",
    "Ubuntu 20.04", "Ubuntu 22.04", "Ubuntu 24.04",
    "iOS 16", "iOS 17", "iOS 18",
    "Android 12", "Android 13", "Android 14", "Android 15",
    "Fedora 39", "Fedora 40",
    "Debian 11", "Debian 12",
    "ChromeOS 120", "ChromeO
    S 125"
]
RESOLUTIONS = ["1920x1080", "1366x768", "2560x1440", "1440x900", "375x812", "390x844"]
EVENT_TYPES = ["page_view", "click", "form_submit", "scroll", "purchase", "add_to_cart", "sign_up"]
TAGS_POOL = ["trending", "recommended", "premium", "new", "featured", "popular", "sale", "limited"]

def generate_record():
    record = {
        "custom_properties": {
            "country": random.choice(COUNTRIES),
            "experiment_id": f"exp_{uuid.uuid4().hex[:6]}",
            "utm_campaign": random.choice(UTM_CAMPAIGNS),
            "utm_medium": random.choice(UTM_MEDIUMS),
            "utm_source": random.choice(UTM_SOURCES),
            "variant": random.choice(VARIANTS)
        },
        "data": uuid.uuid4().hex * 4,
        "device_context": {
            "browser": random.choice(BROWSERS),
            "browser_version": f"{random.randint(100, 125)}.0",
            "device_type": random.choice(DEVICE_TYPES),
            "os": random.choice(OS_LIST),
            "screen_resolution": random.choice(RESOLUTIONS),
            "user_agent": "Mozilla/5.0 (compatible; StreamGen/1.0)"
        },
        "event_timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
        "event_type": random.choice(EVENT_TYPES),
        "page_url": f"https://example.com/app/{uuid.uuid4().hex[:16]}",
        "session_id": str(uuid.uuid4()),
        "tags": random.sample(TAGS_POOL, k=random.randint(1, 4)),
        "user_id": f"user_{uuid.uuid4().hex[:12]}"
    }
    return json.dumps(record)

In [ ]:
click_pct = 70
add_to_cart_pct = 25
purchase_pct = 5
row_count = 10_000_000
other_events = ['page_view', 'form_submit', 'scroll', 'sign_up']
other_pct_each = (100 - click_pct - add_to_cart_pct - purchase_pct) // len(other_events)

In [ ]:
%%sql -r generate_100m_result
ALTER WAREHOUSE MYWH SET WAREHOUSE_SIZE = 'LARGE';

INSERT INTO STREAM.PUBLIC.STREAM1 (DATA)
WITH vals AS (
  SELECT
    ABS(MOD(RANDOM(), 10)) AS r0,
    ABS(MOD(RANDOM(), 5)) AS r1,
    ABS(MOD(RANDOM(), 5)) AS r2,
    ABS(MOD(RANDOM(), 6)) AS r3,
    ABS(MOD(RANDOM(), 4)) AS r4,
    ABS(MOD(RANDOM(), 5)) AS r5,
    ABS(MOD(RANDOM(), 26)) + 100 AS r6,
    ABS(MOD(RANDOM(), 3)) AS r7,
    ABS(MOD(RANDOM(), 13)) AS r8,
    ABS(MOD(RANDOM(), 6)) AS r9,
    ABS(MOD(RANDOM(), 100)) AS r_event,
    ABS(MOD(RANDOM(), 86400)) AS r11,
    ABS(MOD(RANDOM(), 5)) AS r12,
    ABS(MOD(RANDOM(), 3)) AS r13,
    MD5(RANDOM()::VARCHAR) AS id1,
    MD5(RANDOM()::VARCHAR) AS id2
  FROM TABLE(GENERATOR(ROWCOUNT => {{row_count}}))
)
SELECT OBJECT_CONSTRUCT(
  'custom_properties', OBJECT_CONSTRUCT(
    'country', ARRAY_CONSTRUCT('US','FR','DE','GB','JP','BR','IN','CA','AU','MX')[r0],
    'experiment_id', 'exp_' || SUBSTR(id1, 1, 6),
    'utm_campaign', ARRAY_CONSTRUCT('retargeting','brand_awareness','conversion','loyalty','seasonal')[r1],
    'utm_medium', ARRAY_CONSTRUCT('social','email','cpc','organic','referral')[r2],
    'utm_source', ARRAY_CONSTRUCT('email','facebook','google','twitter','linkedin','instagram')[r3],
    'variant', ARRAY_CONSTRUCT('control','variant_a','variant_b','variant_c')[r4]
  ),
  'data', id1 || id2 || id1 || id2,
  'device_context', OBJECT_CONSTRUCT(
    'browser', ARRAY_CONSTRUCT('Chrome','Firefox','Safari','Edge','Opera')[r5],
    'browser_version', r6::VARCHAR || '.0',
    'device_type', ARRAY_CONSTRUCT('desktop','mobile','tablet')[r7],
    'os', ARRAY_CONSTRUCT('Windows 10','Windows 11','macOS 13','macOS 14','Ubuntu 22.04','Ubuntu 24.04','iOS 16','iOS 17','iOS 18','Android 12','Android 13','Android 14','ChromeOS 120')[r8],
    'screen_resolution', ARRAY_CONSTRUCT('1920x1080','1366x768','2560x1440','1440x900','375x812','390x844')[r9],
    'user_agent', 'Mozilla/5.0 (compatible; StreamGen/1.0)'
  ),
  'event_timestamp', TO_VARCHAR(DATEADD('second', -r11, CURRENT_TIMESTAMP()), 'YYYY-MM-DD"T"HH24:MI:SS.FF6"Z"'),
  'event_type', CASE
    WHEN r_event < {{click_pct}} THEN 'click'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} THEN 'add_to_cart'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} THEN 'purchase'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} + {{other_pct_each}} THEN 'page_view'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} + {{other_pct_each}} * 2 THEN 'form_submit'
    WHEN r_event < {{click_pct}} + {{add_to_cart_pct}} + {{purchase_pct}} + {{other_pct_each}} * 3 THEN 'scroll'
    ELSE 'sign_up'
  END,
  'page_url', 'https://example.com/app/' || SUBSTR(id2, 1, 16),
  'session_id', SUBSTR(id1, 1, 8) || '-' || SUBSTR(id1, 9, 4) || '-' || SUBSTR(id1, 13, 4) || '-' || SUBSTR(id2, 1, 4) || '-' || SUBSTR(id2, 5, 12),
  'tags', ARRAY_SLICE(
    ARRAY_CONSTRUCT('trending','recommended','premium','new','featured','popular','sale','limited'),
    r12, r12 + 1 + r13
  ),
  'user_id', 'user_' || SUBSTR(id1, 21, 12)
) AS DATA
FROM vals;

ALTER WAREHOUSE MYWH SET WAREHOUSE_SIZE = 'X-SMALL';

In [ ]:
%%sql -r dataframe_1
ALTER WAREHOUSE MYWH SET WAREHOUSE_SIZE = 'MEDIUM';